# Semantic-aware HNSW Variant C — cosine-safe

This notebook compares HNSW post-filtering, filtered HNSW traversal, and semantic-aware HNSW Variant C.

The Rust prototype originally used `DistDot`, but `anndists` asserts that the dot-product distance input stays non-negative. Real semantic embeddings can have negative cosine similarity, so this notebook patches the prototype to `DistCosine` before compilation. This is the correctness-safe experiment; if the traversal result is promising, the next systems step is a custom normalized-dot distance without per-call norm computation.


In [ ]:
#@title 1) Settings
FULL_DATA = False #@param {type:"boolean"}
QUERIES = 100 #@param {type:"integer"}
K = 50 #@param {type:"integer"}
EF = 128 #@param {type:"integer"}
M = 24 #@param {type:"integer"}
EF_CONSTRUCTION = 200 #@param {type:"integer"}
SEMANTIC_LAMBDA = 0.35 #@param {type:"number"}
GATE_LOGPROB = -1.0 #@param {type:"number"}
BRIDGE_HOPS = 2 #@param {type:"integer"}
BRIDGE_CAP = 64 #@param {type:"integer"}
POSTFILTER_OVERSAMPLE = 8 #@param {type:"integer"}
POSITIVE = 'minimalist,office_appropriate' #@param {type:"string"}
NEGATIVE = 'technical_sporty' #@param {type:"string"}
print(locals().get('FULL_DATA'), QUERIES, K, EF, POSITIVE, NEGATIVE)


In [ ]:
#@title 2) Clone repo + dependencies
import os, pathlib, shutil, subprocess, sys
ROOT = pathlib.Path('/content/ras')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.','faiss-cpu'], check=True)
if shutil.which('rustc') is None or shutil.which('cargo') is None:
    subprocess.run(['bash','-lc', "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"], check=True)
    os.environ['PATH'] = str(pathlib.Path.home()/'.cargo'/'bin') + os.pathsep + os.environ.get('PATH','')
print('commit:', subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip())
print(subprocess.check_output(['rustc','--version']).decode().strip())


In [ ]:
#@title 3) Export real fashion embeddings + compiled semantic programs
import pathlib, shutil, subprocess, sys, time
os.chdir('/content/ras')
CFG = 'configs/binary_bbq.yaml' if FULL_DATA else 'configs/binary_bbq_smoke.yaml'
ASSETS = pathlib.Path('/content/semantic_hnsw_assets')
if ASSETS.exists(): shutil.rmtree(ASSETS)
t0=time.time()
subprocess.run([sys.executable,'-m','experiments.export_native_finalists','--config',CFG,'--out-dir',str(ASSETS)], check=True)
print(f'export finished in {(time.time()-t0)/60:.1f} min')
print('items:', ASSETS/'fp32_items.f32')


In [ ]:
#@title 4) Correctness fix: use cosine-safe HNSW distance
from pathlib import Path
src = Path('/content/ras/rust/semantic_engine/src/bin/semantic_hnsw.rs')
text = src.read_text()
assert 'DistDot' in text or 'DistCosine' in text
text = text.replace('DistDot', 'DistCosine')
src.write_text(text)
print('Rust HNSW distance patched to DistCosine')
print('Reason: normalized semantic embeddings can still have negative pairwise cosine; anndists DistDot asserts dot >= 0.')


In [ ]:
#@title 5) Compile semantic-HNSW
import subprocess, os, time
os.chdir('/content/ras')
t0=time.time()
subprocess.run(['cargo','build','--release','--manifest-path','rust/semantic_engine/Cargo.toml','--bin','semantic_hnsw'], check=True)
BIN='/content/ras/rust/semantic_engine/target/release/semantic_hnsw'
print(f'compiled in {time.time()-t0:.1f}s:', BIN)


In [ ]:
#@title 6) Run C vs baselines
import subprocess, pathlib, time
OUT = pathlib.Path('/content/semantic_hnsw_results.csv')
cmd=[BIN,'--assets',str(ASSETS),'--programs',str(ASSETS/'sidecar_programs'),'--positive',POSITIVE,'--negative',NEGATIVE,'--queries',str(QUERIES),'--k',str(K),'--ef',str(EF),'--m',str(M),'--ef-construction',str(EF_CONSTRUCTION),'--semantic-lambda',str(SEMANTIC_LAMBDA),'--gate-logprob',str(GATE_LOGPROB),'--bridge-hops',str(BRIDGE_HOPS),'--bridge-cap',str(BRIDGE_CAP),'--postfilter-oversample',str(POSTFILTER_OVERSAMPLE),'--out',str(OUT)]
print(' '.join(cmd))
t0=time.time()
run=subprocess.run(cmd,text=True,capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'semantic_hnsw failed with code {run.returncode}')
print(f'total wall time: {time.time()-t0:.1f}s')


In [ ]:
#@title 7) Summary
import pandas as pd, numpy as np
df=pd.read_csv('/content/semantic_hnsw_results.csv')
summary=(df.groupby('method').agg(queries=('query_id','count'),mean_latency_ms=('latency_ms','mean'),p50_latency_ms=('latency_ms','median'),p95_latency_ms=('latency_ms',lambda x: np.quantile(x,.95)),mean_recall_at_k=('recall_at_k','mean'),mean_returned=('returned','mean'),mean_visited=('visited','mean'),mean_semantic_evals=('semantic_evals','mean'),mean_bridge_candidates=('bridge_candidates','mean'),qualified_fraction=('qualified_fraction','mean')).reset_index())
display(summary.sort_values('mean_latency_ms'))
print('NOTE: semantic scores are still precomputed before timed traversal in this prototype.')
